# Task 5 — Source Metadata Ingestion into MongoDB

## Mục tiêu

Đọc metadata từ Kafka bằng Spark Structured Streaming, parse JSON theo schema,
upsert vào MongoDB và tiếp tục đúng offset sau restart.

## Thiết kế và triển khai

[`task5/metadata_stream.py`](https://github.com/linh285/huggingface-cpg-streaming/blob/main/task5/metadata_stream.py) dùng
`StructType`, chỉ nhận `FILE_METADATA_UPSERT` schema `1.0.0`, bổ sung Kafka
partition/offset rồi ghi replace/upsert với `_id=file_id`. Checkpoint
`/opt/spark-checkpoints/cpg-metadata` nằm trên named Docker volume.

## Lệnh thực thi

```bash
python task5/verify_mongodb_corpus.py --timeout 300 \
  --json-output artifacts/task5/mongodb_corpus_verification.json
```

## Kết quả thực nghiệm


In [1]:
import json
from pathlib import Path

proof = json.loads(Path("../artifacts/task5/mongodb_corpus_verification.json").read_text(encoding="utf-8"))
assert proof["status"] == "PASS"
assert proof["total_documents"] == 147
assert proof["distinct_file_ids"] == 147
assert proof["distinct_document_ids"] == 147
assert proof["id_file_id_mismatches"] == 0
assert not proof["missing_manifest_file_ids"]
assert not proof["unexpected_document_ids"]
assert all(value == 0 for value in proof["required_field_missing"].values())
assert proof["spark_query"]["status"] == "RUNNING"
assert proof["checkpoint_file_count"] > 0
print(json.dumps({
    "total_documents": proof["total_documents"],
    "distinct_file_ids": proof["distinct_file_ids"],
    "distinct_document_ids": proof["distinct_document_ids"],
    "id_file_id_mismatches": proof["id_file_id_mismatches"],
    "required_field_missing": proof["required_field_missing"],
    "spark_query": {
        "name": proof["spark_query"]["name"],
        "status": proof["spark_query"]["status"],
    },
    "checkpoint_location": proof["checkpoint_location"],
    "checkpoint_file_count": proof["checkpoint_file_count"],
}, indent=2))


{
  "total_documents": 147,
  "distinct_file_ids": 147,
  "distinct_document_ids": 147,
  "id_file_id_mismatches": 0,
  "required_field_missing": {
    "path": 0,
    "content_sha256": 0,
    "kafka_partition": 0,
    "kafka_offset": 0
  },
  "spark_query": {
    "name": "cpg_metadata_to_mongodb",
    "status": "RUNNING"
  },
  "checkpoint_location": "/opt/spark-checkpoints/cpg-metadata",
  "checkpoint_file_count": 108
}


![Spark Structured Streaming — query RUNNING](images/task5/spark-full-corpus-running.jpg)

![Mongo Express — source_metadata có 147 document](images/task5/mongodb-full-corpus.jpg)

## Vấn đề và cách xử lý

Spark cần tải connector packages ở lần khởi động đầu và từng cạnh tranh bộ nhớ
với Neo4j. Ivy cache, checkpoint volume và giới hạn driver memory giúp restart
nhanh hơn. Mongo Express chỉ được bật ở profile `ui`, không tham gia đường ghi
dữ liệu.

## Đánh giá

Collection có 147 document, 147 `_id`, 147 `file_id`, không thiếu file trong
manifest và đủ `path`, `content_sha256`, `kafka_partition`, `kafka_offset`.
Checkpoint tại `/opt/spark-checkpoints/cpg-metadata` tồn tại và có dữ liệu;
verifier yêu cầu `checkpoint_file_count > 0`. UI chỉ phục vụ quan sát; verifier
dùng truy vấn MongoDB và Spark UI để quyết định kết quả.
